**生成式 AI 使用说明**：本作业中使用生成式 AI 工具时，适用与协作相同的课程政策。和其他协作者一样，每位学生都必须独立写出自己的解答，不能直接依赖交互输出；提交内容中还应注明协作的性质。使用生成式 AI 工具实质性完成作业内容不符合本作业的精神，也会违反 [Honor Code](https://communitystandards.stanford.edu/policies-and-guidance/honor-code)。

In [ ]:
# 将你的 Google Drive 挂载到 Colab VM。
from google.colab import drive
drive.mount('/content/drive')

# TODO：填写 Drive 中保存解压后
# 作业文件夹，例如 'cs231n/assignments/assignment1/'
FOLDERNAME = 'cs231n/assignments/assignment1/'
assert FOLDERNAME is not None, "[!] Enter the foldername."

# 现在已经挂载 Drive，下面确保
# Colab VM 的 Python 解释器可以加载
# 其中的 Python 文件。
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

# 将 CIFAR-10 数据集下载到你的 Drive
# 如果它还不存在。
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/datasets/
!bash get_datasets.sh
%cd /content/drive/My\ Drive/$FOLDERNAME

# Softmax 分类器练习

*请完成并提交这份 worksheet，包括其中的输出以及 worksheet 外部的任何辅助代码。更多细节请参考课程网站上的 [assignments page](http://vision.stanford.edu/teaching/cs231n/assignments.html)。*

在这个练习中，你将：

- 为 Softmax 分类器实现完全向量化的**损失函数**。
- 实现其**解析梯度**的完全向量化表达式。
- 使用数值梯度**检查你的实现**。
- 使用验证集**调优学习率和正则化**强度。
- 使用 **SGD** **优化**损失函数。
- **可视化**最终学到的权重。

In [ ]:
# 运行本 notebook 的初始化代码。
import random
import numpy as np
from cs231n.data_utils import load_CIFAR10
import matplotlib.pyplot as plt

# This is a bit 的 magic 到 make matplotlib 图像 内嵌显示 在 the
# notebook rather than 在 a 新窗口.
%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # 设置图像的默认大小
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# 另一点 magic，用于让 notebook 重新加载外部 Python 模块；
# 参考 http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

## CIFAR-10 数据加载与预处理

In [ ]:
# 加载原始 CIFAR-10 数据。
cifar10_dir = 'cs231n/datasets/cifar-10-batches-py'

# 清理变量，避免多次加载数据导致内存问题
try:
   del X_train, y_train
   del X_test, y_test
   print('Clear previously loaded data.')
except:
   pass

X_train, y_train, X_test, y_test = load_CIFAR10(cifar10_dir)

# 作为合理性检查，打印训练数据和测试数据的大小。
print('Training data shape: ', X_train.shape)
print('Training labels shape: ', y_train.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

In [ ]:
# 可视化 some 样本 来自 数据集.
# We show a few 样本 的 训练 images 来自每个类别.
classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
num_classes = len(classes)
samples_per_class = 7
for y, cls in enumerate(classes):
    idxs = np.flatnonzero(y_train == y)
    idxs = np.random.choice(idxs, samples_per_class, replace=False)
    for i, idx in enumerate(idxs):
        plt_idx = i * num_classes + y + 1
        plt.subplot(samples_per_class, num_classes, plt_idx)
        plt.imshow(X_train[idx].astype('uint8'))
        plt.axis('off')
        if i == 0:
            plt.title(cls)
plt.show()

In [ ]:
# Split 数据 到 训练, val, 并 测试集s. In addition 我们将
# create a sm所有 development set as a subset 的 训练数据;
# we 可以 使用 这个 用于 development so our code runs faster.
num_training = 49000
num_validation = 1000
num_test = 1000
num_dev = 500

# Our 验证 set 将 be num_验证 点 来自 original
# 训练集.
mask = range(num_training, num_training + num_validation)
X_val = X_train[mask]
y_val = y_train[mask]

# Our 训练集 将 be first num_训练 点 来自 original
# 训练集.
mask = range(num_training)
X_train = X_train[mask]
y_train = y_train[mask]

# 我们将 also make a development set, which is a sm所有 subset of
# 训练集.
mask = np.random.choice(num_training, num_dev, replace=False)
X_dev = X_train[mask]
y_dev = y_train[mask]

# 我们使用 first num_测试点 的 original 测试集 as our
# 测试集.
mask = range(num_test)
X_test = X_test[mask]
y_test = y_test[mask]

print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

In [ ]:
# 预处理：把图像数据 reshape 成行向量
X_train = np.reshape(X_train, (X_train.shape[0], -1))
X_val = np.reshape(X_val, (X_val.shape[0], -1))
X_test = np.reshape(X_test, (X_test.shape[0], -1))
X_dev = np.reshape(X_dev, (X_dev.shape[0], -1))

# As a 合理性检查, 打印 形状s 的 数据
print('Training data shape: ', X_train.shape)
print('Validation data shape: ', X_val.shape)
print('Test data shape: ', X_test.shape)
print('dev data shape: ', X_dev.shape)

In [ ]:
# 预处理：减去均值图像
# 第一步：基于训练数据计算图像均值
mean_image = np.mean(X_train, axis=0)
print(mean_image[:10]) # print a few 的 elements
plt.figure(figsize=(4,4))
plt.imshow(mean_image.reshape((32,32,3)).astype('uint8')) # 可视化均值图像
plt.show()

# 第二步：从训练和测试数据中减去均值图像
X_train -= mean_image
X_val -= mean_image
X_test -= mean_image
X_dev -= mean_image

# third: append bias 维度 ones (i.e. bias trick) so 该 our 分类器
# only has 到 worry about optimizing a single weight 矩阵 W.
X_train = np.hstack([X_train, np.ones((X_train.shape[0], 1))])
X_val = np.hstack([X_val, np.ones((X_val.shape[0], 1))])
X_test = np.hstack([X_test, np.ones((X_test.shape[0], 1))])
X_dev = np.hstack([X_dev, np.ones((X_dev.shape[0], 1))])

print(X_train.shape, X_val.shape, X_test.shape, X_dev.shape)

## Softmax 分类器

本节的代码都将写在 `cs231n/classifiers/softmax.py` 中。

你可以看到，我们已经预填了 `softmax_loss_naive` 函数，它使用 for 循环来计算 softmax 损失函数。

In [ ]:
# Evaluate naive 实现 的 损失 we provided 用于 you:
from cs231n.classifiers.softmax import softmax_loss_naive
import time

# generate a random Softmax 分类器 weight 矩阵 的 sm所有 numbers
W = np.random.randn(3073, 10) * 0.0001

loss, grad = softmax_loss_naive(W, X_dev, y_dev, 0.000005)
print('loss: %f' % (loss, ))

# As a rough 合理性检查, our 损失 应为 something 接近 -log(0.1).
print('loss: %f' % loss)
print('sanity check: %f' % (-np.log(0.1)))

**内联问题 1**

为什么我们预期损失会接近 `-log(0.1)`？请简要解释。

$\color{blue}{\textit 你的回答：}$ *在此填写*

上面函数返回的 `grad` 现在全为零。请推导并实现 softmax 损失函数的梯度，并直接在 `softmax_loss_naive` 函数内部实现。把新代码穿插到现有函数中会比较方便。

为了检查梯度是否实现正确，你可以用数值方法估计损失函数的梯度，并与你计算出的解析梯度进行比较。我们已经提供了完成这件事的代码：

In [ ]:
# Once you've implemented 梯度, re计算 it 使用 code below
# and 梯度 check it 使用 函数 we provided 用于 you

# 计算 损失 并 its 梯度 at W.
loss, grad = softmax_loss_naive(W, X_dev, y_dev, 0.0)

# Numeri调用y 计算 梯度 along several randomly chosen 维度, and
# compare them 使用 your analyti调用y 计算得到的 梯度. numbers 应匹配
# almost exactly along 所有 维度.
from cs231n.gradient_check import grad_check_sparse
f = lambda w: softmax_loss_naive(w, X_dev, y_dev, 0.0)[0]
grad_numerical = grad_check_sparse(f, W, grad)

# do 梯度 check once again 使用 正则化 turned on
# you didn't forget 正则化 梯度 did you?
loss, grad = softmax_loss_naive(W, X_dev, y_dev, 5e1)
f = lambda w: softmax_loss_naive(w, X_dev, y_dev, 5e1)[0]
grad_numerical = grad_check_sparse(f, W, grad)

**内联问题 2**

虽然 softmax 损失的梯度检查通常很可靠，但对 SVM 损失来说，偶尔会有某个维度在梯度检查中无法完全匹配。这种差异可能由什么造成？这是否值得担心？请给出一个一维的简单例子，说明 SVM 损失的梯度检查可能失败。改变 margin 会如何影响这种情况发生的频率？

注意，样本 $(x_i, y_i)$ 的 SVM 损失定义为：
$$L_i = \sum_{j\ne y_i}\max(0, s_j - s_{y_i} + \Delta)$$
其中 $j$ 遍历除正确类别 $y_i$ 之外的所有类别，$s_j$ 表示第 $j$ 类的分类器分数，$\Delta$ 是标量 margin。更多信息请参考 [this](https://cs231n.github.io/linear-classify/) 页面中的 “Multiclass Support Vector Machine loss”。

*提示：严格来说，SVM 损失函数并不是处处可微。*

$\color{blue}{\textit 你的回答：}$ *在此填写。*

In [ ]:
# Next implement 函数 softmax_损失_vectorized; 用于 now only 计算 损失;
# 我们将 implement 梯度 在 a moment.
tic = time.time()
loss_naive, grad_naive = softmax_loss_naive(W, X_dev, y_dev, 0.000005)
toc = time.time()
print('Naive loss: %e computed in %fs' % (loss_naive, toc - tic))

from cs231n.classifiers.softmax import softmax_loss_vectorized
tic = time.time()
loss_vectorized, _ = softmax_loss_vectorized(W, X_dev, y_dev, 0.000005)
toc = time.time()
print('Vectorized loss: %e computed in %fs' % (loss_vectorized, toc - tic))

# 损失 应匹配 but your vectorized 实现 应为 much faster.
print('difference: %f' % (loss_naive - loss_vectorized))

In [ ]:
# Complete 实现 的 softmax_损失_vectorized, 并 计算 梯度
# of 损失 函数 在 a vectorized way.

# naive 实现 并 vectorized 实现 应匹配, but
# 向量化版本 应该 still be much faster.
tic = time.time()
_, grad_naive = softmax_loss_naive(W, X_dev, y_dev, 0.000005)
toc = time.time()
print('Naive loss and gradient: computed in %fs' % (toc - tic))

tic = time.time()
_, grad_vectorized = softmax_loss_vectorized(W, X_dev, y_dev, 0.000005)
toc = time.time()
print('Vectorized loss and gradient: computed in %fs' % (toc - tic))

# 损失 is a single number, so it is easy 到 compare 值 计算得到的
# by two 实现s. 梯度 on other hand is a 矩阵, so
# 我们使用 Frobenius norm 到 compare them.
difference = np.linalg.norm(grad_naive - grad_vectorized, ord='fro')
print('difference: %f' % difference)

### 随机梯度下降

现在我们已经有了损失和梯度的向量化高效表达式，并且梯度与数值梯度匹配。因此，可以开始用 SGD 最小化损失。本部分代码将写在 `cs231n/classifiers/linear_classifier.py` 中。

In [ ]:
# In file linear_分类器.py, implement SGD 在 函数
# LinearClassifier.训练() 并 然后 run it 使用 code below.
from cs231n.classifiers import Softmax
softmax = Softmax()
tic = time.time()
loss_hist = softmax.train(X_train, y_train, learning_rate=1e-7, reg=2.5e4,
                      num_iters=1500, verbose=True)
toc = time.time()
print('That took %fs' % (toc - tic))

In [ ]:
# A 使用ful debugging strategy is 到 plot 损失 as a 函数 of
# iteration number:
plt.plot(loss_hist)
plt.xlabel('Iteration number')
plt.ylabel('Loss value')
plt.show()

In [ ]:
# Write LinearClassifier.predict 函数 并 evaluate performance on
# both 训练 并 验证 set
# 你应该 get 验证 accuracy 的 about 0.34 (> 0.33).
y_train_pred = softmax.predict(X_train)
print('training accuracy: %f' % (np.mean(y_train == y_train_pred), ))
y_val_pred = softmax.predict(X_val)
print('validation accuracy: %f' % (np.mean(y_val == y_val_pred), ))

In [ ]:
# 保存训练好的模型，供 autograder 使用。
softmax.save("softmax.npy")

In [ ]:
# 使用 验证 set 到 调优 hyper参数 (正则化 strength and
# 学习率). 你应该 experiment 使用 different ranges 用于 learning
# rates 并 正则化 strengths; if you are careful you 应为 able to
# get a 分类 accuracy 的 about 0.365 (> 0.36) on 验证 set.

# 注意： 你可以 see runtime/在flow warnings during hyper-parameter search.
# This may be ca使用 by extreme 值, 并 is not a bug.

# 结果 is 字典 mapping tuples 的 form
# (learning_rate, 正则化_strength) 到 tuples 的 form
# (训练_accuracy, 验证_accuracy). accuracy is simply fraction
# of 数据 点 该 are correctly 类别ified.
results = {}
best_val = -1   # highest 验证 accuracy 该 我们已经 seen so far.
best_softmax = None # Softmax object 该 achieved highest 验证 rate.

################################################################################
# TODO:                                                                        #
# Write code 该 chooses best hyper参数 by 调优 on 验证 #
# set. 对于每个 combination 的 hyper参数, 训练 a Softmax on the.        #
# 训练集, 计算 its accuracy on 训练 并 验证 sets, 并  #
# 存储 这些 numbers 在 结果 字典. In addition, 存储 best   #
# 验证 accuracy 在 best_val 并 Softmax object 该 achieves 这个.   #
# accuracy 在 best_softmax.                                                    #
#                                                                              #
# 提示： 你应该 使用 a sm所有 值 用于 num_iters as you develop your         #
# 验证 code so 该 分类器s don't take much time 到 训练; once  #
# you are confident 该 your 验证 code works, 你应该 rerun      #
# code 使用 a larger 值 用于 num_iters.                                      #
################################################################################

# Provided as a 参考. 你可以 or may not want 到 change 这些 hyper参数
learning_rates = [1e-7, 1e-6]
regularization_strengths = [2.5e4, 1e4]



# 打印结果。
for lr, reg in sorted(results):
    train_accuracy, val_accuracy = results[(lr, reg)]
    print('lr %e reg %e train accuracy: %f val accuracy: %f' % (
                lr, reg, train_accuracy, val_accuracy))

print('best validation accuracy achieved during cross-validation: %f' % best_val)

In [ ]:
# 可视化 cross-验证 结果
import math
import pdb

# pdb.set_trace()

x_scatter = [math.log10(x[0]) for x in results]
y_scatter = [math.log10(x[1]) for x in results]

# plot 训练准确率
marker_size = 100
colors = [results[x][0] for x in results]
plt.subplot(2, 1, 1)
plt.tight_layout(pad=3)
plt.scatter(x_scatter, y_scatter, marker_size, c=colors, cmap=plt.cm.coolwarm)
plt.colorbar()
plt.xlabel('log learning rate')
plt.ylabel('log regularization strength')
plt.title('CIFAR-10 training accuracy')

# plot 验证 accuracy
colors = [results[x][1] for x in results] # default size 的 markers is 20
plt.subplot(2, 1, 2)
plt.scatter(x_scatter, y_scatter, marker_size, c=colors, cmap=plt.cm.coolwarm)
plt.colorbar()
plt.xlabel('log learning rate')
plt.ylabel('log regularization strength')
plt.title('CIFAR-10 validation accuracy')
plt.show()

In [ ]:
# Evaluate best softmax on 测试集
y_test_pred = best_softmax.predict(X_test)
test_accuracy = np.mean(y_test == y_test_pred)
print('Softmax classifier on raw pixels final test set accuracy: %f' % test_accuracy)

In [ ]:
# 保存最佳 softmax 模型
best_softmax.save("best_softmax.npy")

In [ ]:
# 可视化每个类别学到的权重。
# Depending on your choice 的 学习率 并 正则化 strength, 这些 may
# or may not be nice 到 look at.
w = best_softmax.W[:-1,:] # 去掉 bias
w = w.reshape(32, 32, 3, 10)
w_min, w_max = np.min(w), np.max(w)
classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
for i in range(10):
    plt.subplot(2, 5, i + 1)

    # 将权重缩放到 0 到 255 之间
    wimg = 255.0 * (w[:, :, :, i].squeeze() - w_min) / (w_max - w_min)
    plt.imshow(wimg.astype('uint8'))
    plt.axis('off')
    plt.title(classes[i])

**内联问题 3**

描述你可视化出来的 Softmax 分类器权重看起来是什么样子，并简要解释为什么会呈现这种形态。

$\color{blue}{\textit 你的回答：}$ *在此填写*

**内联问题 4** - *True or False*

假设总训练损失定义为所有训练样本逐样本损失的总和。是否可能向训练集中加入一个新样本，使 softmax 损失发生变化，但 SVM 损失保持不变？

$\color{blue}{\textit 你的回答：}$


$\color{blue}{\textit 你的解释：}$